# Milestone 11 — Perplexity Benchmark

wikitext-2-raw-v1 기준 perplexity 측정.

| 모델 | 측정 대상 |
|------|----------|
| FP16 (baseline) | 기준선 |
| W4A16 (RTN) | weight-only INT4 |
| W8A8 (static) | weight+activation INT8 |
| W8A8 dynamic | per-token dynamic INT8 |

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import math
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from mini_compressor import Compressor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_ID = 'Qwen/Qwen3-0.6B'
STRIDE   = 512
MAX_LEN  = 2048
print(f'device: {device}')

In [ ]:
# wikitext-2 로드 및 토크나이즈
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
text = '\n\n'.join(dataset['text'])
encodings = tokenizer(text, return_tensors='pt')
print(f'토큰 수: {encodings.input_ids.shape[1]:,}')

In [ ]:
def compute_perplexity(model, encodings, stride=STRIDE, max_len=MAX_LEN, device=device):
    """sliding window perplexity (Hugging Face 공식 방식)."""
    input_ids = encodings.input_ids.to(device)
    seq_len = input_ids.shape[1]

    nlls = []
    prev_end = 0

    for begin in range(0, seq_len, stride):
        end = min(begin + max_len, seq_len)
        target_len = end - prev_end

        input_chunk  = input_ids[:, begin:end]
        target_chunk = input_ids[:, begin:end].clone()
        # 이전 context 부분은 loss 계산에서 제외
        target_chunk[:, :-target_len] = -100

        with torch.no_grad():
            loss = model(input_chunk, labels=target_chunk).loss
        nlls.append(loss.item() * target_len)
        prev_end = end
        if end == seq_len:
            break

    ppl = math.exp(sum(nlls) / prev_end)
    return ppl

## 1. FP16 baseline

In [ ]:
model_fp = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(device)
model_fp.eval()

ppl_fp = compute_perplexity(model_fp, encodings)
print(f'[FP16]  PPL = {ppl_fp:.2f}')
del model_fp
torch.cuda.empty_cache()

## 2. W4A16 (RTN)

In [ ]:
model_w4a16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(device)
model_w4a16.eval()

Compressor.from_recipe('w4a16', targets=['Linear'], ignore=['lm_head']).compress(model_w4a16)

ppl_w4a16 = compute_perplexity(model_w4a16, encodings)
print(f'[W4A16] PPL = {ppl_w4a16:.2f}  (vs FP16: +{ppl_w4a16 - ppl_fp:.2f})')
del model_w4a16
torch.cuda.empty_cache()

## 3. W8A8 static

In [ ]:
model_w8a8 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(device)
model_w8a8.eval()

# calibration 데이터 — wikitext-2 train split 128 샘플
calib_dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
calib_texts = [t for t in calib_dataset['text'] if len(t.strip()) > 50][:128]
calib_inputs = [
    {k: v.to(device) for k, v in tokenizer(t, return_tensors='pt', truncation=True, max_length=512).items()}
    for t in calib_texts
]

Compressor.from_recipe('w8a8', targets=['Linear'], ignore=['lm_head']).compress(model_w8a8, dataloader=calib_inputs)

ppl_w8a8 = compute_perplexity(model_w8a8, encodings)
print(f'[W8A8]  PPL = {ppl_w8a8:.2f}  (vs FP16: +{ppl_w8a8 - ppl_fp:.2f})')
del model_w8a8
torch.cuda.empty_cache()

## 4. W8A8 dynamic (per-token)

In [ ]:
model_dynamic = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(device)
model_dynamic.eval()

Compressor.from_recipe('w8a8_dynamic', targets=['Linear'], ignore=['lm_head']).compress(model_dynamic)

ppl_dynamic = compute_perplexity(model_dynamic, encodings)
print(f'[W8A8-dynamic] PPL = {ppl_dynamic:.2f}  (vs FP16: +{ppl_dynamic - ppl_fp:.2f})')
del model_dynamic
torch.cuda.empty_cache()

## 결과 요약

In [ ]:
results = {
    'FP16 (baseline)': ppl_fp,
    'W4A16 RTN':       ppl_w4a16,
    'W8A8 static':     ppl_w8a8,
    'W8A8 dynamic':    ppl_dynamic,
}

print('=' * 45)
print(f'{"Scheme":<22} {"PPL":>8}  {"Δ vs FP16":>10}')
print('-' * 45)
for name, ppl in results.items():
    delta = f'+{ppl - ppl_fp:.2f}' if ppl != ppl_fp else '—'
    print(f'{name:<22} {ppl:>8.2f}  {delta:>10}')
print('=' * 45)